# Pixel explorer: standing cane test AOI (2025 or 2026 season, S1 + S2)

Set `SEASON = 2025` (labels / QC season) or `SEASON = 2026` (current season: model maps 2025 / 2026 and the change class are shown in section 0).

Give a **pixel id** (band 1 of `data/grid/index.tif`, open it in QGIS and read the value) or lon/lat, then run all cells.

0. **Label / QC** (optional, `SHOW_QC`): 2025 label, clean label, QC flags, out-of-fold model p(cane).
1. **15-day S1 composites** (the model inputs) with the median clean cane / non-cane curves (dashed / dotted).
2. **NDVI**: raw, cloud-masked and Whittaker-smoothed.
3. **Sentinel-1** VH / VV on every acquisition (no rain data in this project).
4. **S2 chips** on the dates this pixel is cloud-free (red box = the pixel).

Blind-review sample points (200, for QGIS): `data/labels/qc_review/key/samples.parquet`; sheets in `data/labels/qc_review/sheets/`.

In [ ]:
%matplotlib inline
import sys
sys.path.insert(0, "../scripts")
import importlib, explorer as ex
importlib.reload(ex)
import pandas as pd
from IPython.display import display

## Parameters

In [ ]:
SITE = "site82_2026"
PIXEL_ID = 8063680
SEASON = 2026
SAMPLE_ID = "R001"          # a blind-review sample (R001-R200); used only when PIXEL_ID is None and LON is None
LON, LAT = None, None       # e.g. 102.05, 16.30
RADIUS = 0                  # 0 = single pixel; 1 = 3x3 window mean (S1 averaged in linear power)

START, END = ("2025-11-01", "2026-10-31") if SEASON == 2026 else ("2024-09-01", "2025-11-30")   # date range
BACKEND = "matplotlib"          # "plotly" (interactive) or "matplotlib"
SHOW_QC = True              # False = look blind (no label / flags / model output)

LAMBDA, D = 1.0, 2          # Whittaker smoothing
CHIP_SIZE, CHIP_BANDS = 64, ("B5", "B3", "B2")   # NIR false colour; ("B5","B3","B2") or ("B4","B3","B2") also work
# STRETCH (how chip colours are scaled):
#   "fixed" = the same range for every date (AOI-wide 2-98 % per band, or a default range): colours are
#             comparable between dates -> best to follow a field through the season (default)
#   "chip"  = each chip stretched on its own 2-98 %: more contrast, but colours NOT comparable between dates
#             (good for hazy / dark dates)
#   {"B8": (1000, 4500), "B4": (200, 2200), "B3": (300, 1800)} = your own fixed range per band
#             (surface reflectance x 10000; bands missing from the dict fall back to the per-chip stretch)
# CHIP_BANDS: ("B8","B4","B3") NIR false colour (vegetation red) | ("B5","B3","B2") 5-3-2 | ("B4","B3","B2") true colour
STRETCH, CHIP_COLUMNS = "chip", 6

## Load the pixel

In [ ]:
from pathlib import Path
if SITE == "site119":
    ex.use_season(SEASON)
    kf = "../sites/2026/province_cp/site119_2026/review_review2026/key/key.csv" if SEASON == 2026 else "../data/labels/qc_review/key/key.csv"
else:
    ex.use_site(SITE, SEASON)
    kf = next(iter(sorted(Path(f"../sites/{SITE}").glob("review*/key/key.csv"))), None)
key = pd.read_csv(kf) if kf else pd.DataFrame(columns=["sample_id", "pixel_id", "lon", "lat", "group"])
print("pixel ids:", ex.GRID)
if PIXEL_ID is None and LON is None:
    PIXEL_ID = int(key.set_index("sample_id").loc[SAMPLE_ID, "pixel_id"])
px = ex.load_pixel(pixel_id=PIXEL_ID, lon=LON, lat=LAT, radius=RADIUS)
print(ex._title(px))
print(f"S2 dates: {len(px['s2'])}, cloud-free at this pixel: {int(px['s2'].clear.sum())}   |   S1 acquisitions: "
      + ", ".join(f"{t} {n}" for t, n in px["s1"].track.value_counts().items()))

## 1. Sentinel-2 chips on cloud-free dates (red box = the pixel)

In [ ]:
ex.plot_chips(px, START, END, size=CHIP_SIZE, bands=CHIP_BANDS, stretch=STRETCH, ncols=CHIP_COLUMNS, backend=BACKEND);

## 2. Label and QC result (hidden when SHOW_QC = False)

In [ ]:
if SHOW_QC:
    display(pd.Series(px["qc"], name="value").to_frame())
    hit = key[key.pixel_id == px["pixel_id"]]
    if len(hit):
        display(hit)

## 3. 15-day S1 composites (model inputs)

In [ ]:
ex.plot_15d(px, show_ref=True);

## 4. NDVI: raw, cloud-masked, Whittaker

In [ ]:
ex.plot_ndvi(px, START, END, backend=BACKEND, lam=LAMBDA, d=D);

## 5. Sentinel-1 VH / VV, every acquisition

In [ ]:
ex.plot_s1(px, START, END, backend=BACKEND, show_daily_rain=True);

## Review sample list (pixel ids to try)

In [ ]:
cols = ["sample_id", "pixel_id", "lon", "lat"] + (["group", "label_2025", "p_oof"] if SHOW_QC else [])
display(key[cols].sort_values("sample_id").head(200))

## Data notes

| Item | Detail |
|---|---|
| Grid | EPSG:32648, 10 m, 4019 x 3503 px; `pixel_id = row * 4019 + col`; band 2 of `index.tif` = inside AOI |
| Sentinel-2 | L2A, 66 dates 2024-09-02..2025-11-26 (dates with AOI cloud > 75 % not downloaded), B2 B3 B4 B5 B8 |
| Cloud-free rule | Cloud Score+ cs_cdf >= 0.60 and SCL / MSK_CLASSI cloud classes off (`data/stacks/S2_CLEAR.tif`) |
| Sentinel-1 | GRD IW VV/VH dB, RO164 descending + RO099 ascending; 15-day composites = 3x3 mean in linear power |
| QC raster | `data/labels/cane2025_qc.tif`: label_clean, flags, block_id (500 m), unit_id, p_oof, d_unit, edge_px |
| QC flags | 1 outlier_unit, 2 water, 4 built, 8 tree, 16 bare, 32 s2_not_green, 64 unit_bad, 128 cl_cane_low, 256 cl_noncane_high |